# 07. 실측 유전자 섭동 반응 예측

05의 Geneformer 가상 삭제는 embedding 변화 탐색입니다. 이 노트북은 실제 control/treated 발현량으로 평가한 별도 task를 읽습니다. 기본 동작은 저장된 결과 읽기이며 GEARS 전체 학습·원본 다운로드를 시작하지 않습니다.


In [1]:
from pathlib import Path
import json, sys, subprocess
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
REPORT = ROOT / 'reports/research_phase2'


## 데이터와 평가 단위

Norman은 공식 전처리 K562 CRISPRa 데이터와 GEARS simulation seed 1 split입니다. HepG2는 GSE264667의 day-7 CRISPRi 데이터입니다. 서로 다른 세포주·개입이므로 하나의 점수로 합치지 않습니다. HepG2의 dual sgRNA는 동일 유전자 표적 설계이며 두 유전자 조합 KO로 취급하지 않습니다. 모든 guide/transcript 변형을 target 유전자별로 묶어 holdout했습니다.

HepG2의 기술 배치(gem_group)마다 non-targeting control을 제공합니다. biological replicate는 unknown으로 기록했습니다. 출력 유전자 2,000개는 학습 target 세포와 제공된 control만 이용해 고릅니다. 평균 발현량과 변화량은 log1p(10k-normalized counts) 공간에서 비교합니다. 성인 HCC 환자 약효나 임상 반응으로 해석하지 않습니다.


In [2]:
contract = json.loads((REPORT / 'hepg2_data_contract/contract.json').read_text())
print('eligible target counts:', contract['target_split_counts'])
print('control policy:', contract['control_policy'])
display(pd.read_csv(REPORT / 'hepg2_data_contract/batch_control_support.tsv', sep='\t').head())


eligible target counts: {'train': 1714, 'test': 214, 'val': 214}
control policy: Non-targeting controls from the same gem_group are available for training and evaluation; held-out treated responses never enter training


,batch,n_non_targeting_controls,biological_replicate
0,3,116,unknown
1,12,91,unknown
2,53,92,unknown
3,39,91,unknown
4,24,75,unknown


## 간단한 예측기와 실제 측정값 비교

B00은 변화가 없다는 matched control 평균, B01은 학습 intervention의 평균 변화, B02는 외부 GO membership에서 학습 반응 PCA 계수를 예측하는 ridge입니다. B02의 PCA는 학습 response만 사용하고 alpha는 validation MSE로 선택합니다. 새로운 target의 실측 response를 intervention feature로 사용하지 않습니다.

MSE와 Δ correlation은 intervention별 평균을 동일 가중치로 합칩니다. No-change의 Δ correlation은 정의되지 않으므로 NaN입니다. 원 발현량 correlation이 높아도 변화 예측이 좋다는 뜻은 아닙니다. `observed_top20_delta`는 평가 시 실측 평균 변화의 절댓값 상위 20개를 보는 보조 지표이며 predictor 입력·선택에는 사용하지 않았습니다.


In [3]:
for name in ['B00_B02_norman_simulation_seed1_v2', 'B00_B02_hepg2_unseen_target_seed42_v1']:
    print(name)
    display(pd.read_csv(REPORT / 'runs' / name / 'metrics.tsv', sep='\t'))


B00_B02_norman_simulation_seed1_v2


,model_id,n_cells,mse,mae,pearson_expression,pearson_delta,observed_mean_delta_l2,mse_observed_top20_delta,sign_accuracy_observed_top20_delta
0,B00_no_change,268.728972,0.006831,0.022436,0.979402,NaN,5.391163,0.642751,0.000000
1,B01_train_mean,268.728972,0.004989,0.019206,0.984842,0.578628,5.391163,0.475942,0.870561
2,B02_GO_PCA_ridge,268.728972,0.002839,0.015579,0.991331,0.703796,5.391163,0.235853,0.891589


B00_B02_hepg2_unseen_target_seed42_v1


,model_id,n_cells,mse,mae,pearson_expression,pearson_delta,observed_mean_delta_l2,mse_observed_top20_delta,sign_accuracy_observed_top20_delta
0,B00_no_change,65.392523,0.019060,0.093090,0.956944,NaN,5.533477,0.304628,0.000000
1,B01_train_mean,65.392523,0.016177,0.089355,0.962942,0.324815,5.533477,0.226941,0.734813
2,B02_GO_PCA_ridge,65.392523,0.012566,0.080464,0.971014,0.394152,5.533477,0.152947,0.784813


## GEARS 실행 상태

GEARS는 실제 control/treated graph를 학습하는 별도 모델입니다. 아래는 완료된 run.json이 있을 때만 점수를 표시합니다. 공식 Norman은 원 DE 처리와 simulation split을 유지하는 재현이며, HepG2 strict 정책과 별도 관리합니다. 학습 중인 작업을 완료로 표시하지 않습니다.


In [4]:
path = REPORT / 'runs/R01_norman_simulation_seed1_v2/run.json'
if path.exists():
    gears = json.loads(path.read_text())
    print(gears['status'], gears['metrics'])
else:
    print('완료된 GEARS run.json이 아직 없습니다. 이 노트북은 학습을 시작하지 않습니다.')


완료된 GEARS run.json이 아직 없습니다. 이 노트북은 학습을 시작하지 않습니다.


## 선택적인 CPU 기준선 재실행

이미 준비된 HepG2 task와 GO prior를 사용합니다. 원본 5GB 파일 변환이나 전체 GEARS 학습을 다시 수행하지 않습니다. 기존 결과를 보존하기 위해 새 run ID를 만듭니다.


In [5]:
RUN_BASELINES = False
if RUN_BASELINES:
    from datetime import datetime, timezone
    run_id = 'tutorial07_hepg2_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
    subprocess.run([sys.executable, str(ROOT / 'scripts/research/run_response_baselines.py'), '--task', 'hepg2', '--run-id', run_id], cwd=ROOT, check=True)
    display(pd.read_csv(REPORT / 'runs' / run_id / 'metrics.tsv', sep='\t'))
else:
    print('저장 결과 읽기 모드')


저장 결과 읽기 모드
